# Bank Customer Churn Prediction

This notebook implements an end-to-end machine learning pipeline for predicting customer churn. It includes data preprocessing, feature engineering, model training using XGBoost, and a detailed performance evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# Set seeds for deterministic results
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    
set_seeds()

## 1. Data Acquisition and Cleaning
We utilize the Churn Modelling dataset. The data is loaded directly from a raw source to ensure reproducibility across different environments.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/aditya-parmar/Bank-Churn-Prediction/master/Churn_Modelling.csv"
df = pd.read_csv(DATA_URL)

print(f"Dataset Size: {df.shape}")
df.head()

In [ ]:
# Dropping administrative identifiers to prevent data leakage
df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Feature Engineering: Categorical encoding
df_processed = pd.get_dummies(df_clean, columns=['Geography', 'Gender'], drop_first=True)

print(f"Processed Feature Set: {df_processed.columns.tolist()}")

## 2. Model Development
XGBoost is selected as the primary estimator due to its robustness with tabular data and efficient handling of feature interactions.

In [ ]:
X = df_processed.drop('Exited', axis=1)
y = df_processed['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train_scaled, y_train)

print("Model training complete.")

## 3. Evaluation and Qualitative Analysis
Performance is assessed using F1-score to maintain a balance between precision and recall, which is critical for imbalanced classes.

In [ ]:
y_pred = model.predict(X_test_scaled)
print("Classification Metrics:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Performance Heatmap (Confusion Matrix)')
plt.show()

### Error Diagnostics
Analyzing False Negatives to identify underlying feature distributions where the model struggles to identify churn.

In [ ]:
results = X_test.copy()
results['Actual'] = y_test
results['Predicted'] = y_pred

false_negatives = results[(results['Actual'] == 1) & (results['Predicted'] == 0)]
print(f"False Negatives Count: {len(false_negatives)}")
false_negatives.head()